<a href="https://colab.research.google.com/github/kudinha/4100-Lab-Dialysis/blob/code/Dialysis_day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script solves for questions 1-3 in Lab Day 2 discussions.

Imports/loading data

In [ ]:
import numpy as np
import pandas as pd

GPH_TO_LPS = 3.78541 / 3600
STREAMS = ["Tube In", "Tube Out", "Shell In", "Shell Out"]
V_COLS = ["Tube_In_A0_V", "Tube_Out_A1_V", "Shell_In_A2_V", "Shell_Out_A3_V"]

T_C = 25                 # deg C — update to your measured temperature
NU = 0.893e-6            # kinematic viscosity of water at 25 C [m^2/s]
D_AB = 1.99e-9           # KCl in water, infinite dilution, 25 C [m^2/s]

SC = NU / D_AB           # Schmidt number, eqn [5]

Equations 9-11

In [ ]:
def logmeanconc(c):
    """Counter-current log-mean concentration difference, eqn [10]."""
    c_ri, c_ro, c_di, c_do = c
    return ((c_ro - c_di) - (c_ri - c_do)) / np.log((c_ro - c_di) / (c_ri - c_do))

def transfer_solute(q_ri, q_ro, c_ri, c_ro):
    """Solute transfer out of feed stream, eqn [11]. Q in L/s, C in mol/L -> mol/s."""
    return q_ri * c_ri - q_ro * c_ro

def k_a_calc(w_r, area, dc_lm):
    """Overall dialysis coefficient, eqn [9]. Converts mol/L -> mol/m^3 so K_A is in m/s."""
    return w_r / (area * dc_lm * 1000)

Data loading

In [ ]:
def load_file(path, n_ss=30):
    """One row per run: steady-state voltages (last n_ss points) + flows from Summary."""
    sheets = pd.read_excel(path, sheet_name=None)
    rows = []
    for _, s in sheets["Summary"].iterrows():
        tail = sheets[f"RUN {int(s['Run Number'])}"][V_COLS].tail(n_ss)
        row = {"file": path.split("/")[-1], "solute": s["Solute"], "run": int(s["Run Number"])}
        for st, col in zip(STREAMS, V_COLS):
            row[f"V {st}"] = tail[col].mean()
            row[f"Q {st}"] = s[f"Total {st} Flow (GPH)"] * GPH_TO_LPS
            row[f"dQ {st}"] = s[f"Total {st} Error (GPH)"] * GPH_TO_LPS
        rows.append(row)
    return pd.DataFrame(rows)

Conductivity calibration

In [ ]:
C_REF = 0.1  # M
water = load_file("/content/DI_0M_10-05.xlsx").iloc[0]
kcl = load_file("/content/KCl_0.1M_10-05.xlsx").iloc[0]
V_WATER = np.array([water[f"V {st}"] for st in STREAMS])
V_REF = np.array([kcl[f"V {st}"] for st in STREAMS])

def volts_to_conc(V):
    """Per-probe linear interpolation, water -> 0.1 M. V is (n_runs, 4) in STREAMS order."""
    return C_REF * (V - V_WATER) / (V_REF - V_WATER)

pd.DataFrame({"V water": V_WATER, "V 0.1 M": V_REF}, index=STREAMS)

,V water,V 0.1 M
Tube In,0.0,1.672543
Tube Out,0.0,1.889213
Shell In,0.0,1.894440
Shell Out,0.0,1.952233


Load runs/convert to concentration

In [ ]:
FILES = ["/content/KCl-245-5_0.1M_10-05.xlsx"]   # add more files here
df = pd.concat([load_file(f) for f in FILES], ignore_index=True)

C = volts_to_conc(df[[f"V {st}" for st in STREAMS]].values)   # mol/L
Q = df[[f"Q {st}" for st in STREAMS]].values                   # L/s
dQ = df[[f"dQ {st}" for st in STREAMS]].values
for j, st in enumerate(STREAMS):
    df[f"C {st}"] = C[:, j]

df[["file", "run"] + [f"C {st}" for st in STREAMS]].round(4)

,file,run,C Tube In,C Tube Out,C Shell In,C Shell Out
0,KCl-245-5_0.1M_10-05.xlsx,1,0.1023,0.0216,0.0,0.0800
1,KCl-245-5_0.1M_10-05.xlsx,2,0.1024,0.0227,0.0,0.0813
2,KCl-245-5_0.1M_10-05.xlsx,3,0.1024,0.0098,0.0,0.0611
3,KCl-245-5_0.1M_10-05.xlsx,4,0.1025,0.0039,0.0,0.0400
4,KCl-245-5_0.1M_10-05.xlsx,5,0.1025,0.0118,0.0,0.0660
5,KCl-245-5_0.1M_10-05.xlsx,6,0.1028,0.0523,0.0,0.0960


Material balance

In [ ]:
#errors are percent error for everything
df["N in"] = Q[:, 0] * C[:, 0] + Q[:, 2] * C[:, 2]
df["N out"] = Q[:, 1] * C[:, 1] + Q[:, 3] * C[:, 3]
df["dN"] = np.sqrt(((dQ * C) ** 2).sum(axis=1))        # propagated flow error
df["closure %"] = 100 * (df["N out"] - df["N in"]) / df["N in"]
df["closes?"] = (df["N out"] - df["N in"]).abs() <= df["dN"]
df["Q tube imbalance %"] = 100 * (Q[:, 1] - Q[:, 0]) / Q[:, 0]
df["Q shell imbalance %"] = 100 * (Q[:, 3] - Q[:, 2]) / Q[:, 2]

df[["run", "N in", "N out", "dN", "closure %", "closes?",
    "Q tube imbalance %", "Q shell imbalance %"]].round(6)

,run,N in,N out,dN,closure %,closes?,Q tube imbalance %,Q shell imbalance %
0,1,0.000215,0.000214,0.000014,-0.695006,True,0.000000,0.000000
1,2,0.000215,0.000210,0.000014,-2.395068,True,0.000000,-5.000000
2,3,0.000210,0.000209,0.000014,-0.574282,True,12.820513,0.000000
3,4,0.000205,0.000199,0.000013,-2.628970,True,26.315789,-10.000000
4,5,0.000215,0.000198,0.000014,-8.064371,False,0.000000,-3.846154
5,6,0.000216,0.000191,0.000013,-11.765986,False,0.000000,-20.000000


Dialysis coefficient

In [ ]:
A = None  # membrane area [m^2] — fill in

df["W_R (mol/s)"] = transfer_solute(Q[:, 0], Q[:, 1], C[:, 0], C[:, 1])
df["dC_lm (M)"] = logmeanconc(C.T)
df["K_A*A (m^3/s)"] = df["W_R (mol/s)"] / (df["dC_lm (M)"] * 1000)
if A:
    df["K_A (m/s)"] = k_a_calc(df["W_R (mol/s)"], A, df["dC_lm (M)"])

df[["run", "W_R (mol/s)", "dC_lm (M)", "K_A*A (m^3/s)"] + (["K_A (m/s)"] if A else [])]

,run,W_R (mol/s),dC_lm (M),K_A*A (m^3/s)
0,1,0.000170,0.021931,0.000008
1,2,0.000168,0.021889,0.000008
2,3,0.000187,0.021871,0.000009
3,4,0.000195,0.021149,0.000009
4,5,0.000191,0.021848,0.000009
5,6,0.000106,0.022228,0.000005


Dialyzer Geometry

In [ ]:
DIALYZERS = {
    "Optiflux F16NR": dict(L=0.260, A=1.5, shell_od=42.9e-3, shell_t=3e-3),
    "Rexeed 21S":     dict(L=0.290, A=2.1, shell_od=46.9e-3, shell_t=3e-3),
}
DIALYZER = "Optiflux F16NR"   # <-- set to the module you used

g = DIALYZERS[DIALYZER]
L, A = g["L"], g["A"]                 # fiber length [m], internal membrane area [m^2]
D_I = 0.200e-3                         # fiber ID [m]
Z = 0.045e-3                           # membrane thickness [m]
r_i = D_I / 2
r_o = r_i + Z
D_O = 2 * r_o

N_FIB = A / (np.pi * D_I * L)          # number of fibers
D_SHELL = g["shell_od"] - 2 * g["shell_t"]   # shell ID [m]

pd.Series({"fibers": N_FIB, "shell ID (m)": D_SHELL, "r_i (m)": r_i, "r_o (m)": r_o})

,0
fibers,9182.015948
shell ID (m),0.036900
r_i (m),0.000100
r_o (m),0.000145


Shell-side hydraulic diameter

In [ ]:
A_C = np.pi / 4 * (D_SHELL**2 - N_FIB * D_O**2)   # shell flow area [m^2]
P_WET = N_FIB * np.pi * D_O                        # wetted perimeter of all fibers [m]
D_H = 4 * A_C / P_WET

pd.Series({"A_c (m^2)": A_C, "P (m)": P_WET, "D_H (m)": D_H})

,0
A_c (m^2),0.000463
P (m),8.365385
D_H (m),0.000221


K_a from 2 a and b

In [ ]:
df["K_A (m/s)"] = k_a_calc(df["W_R (mol/s)"], A, df["dC_lm (M)"])
df[["run", "K_A (m/s)"]]

,run,K_A (m/s)
0,1,0.000005
1,2,0.000005
2,3,0.000006
3,4,0.000006
4,5,0.000006
5,6,0.000003


Equation 5: velocities and reynolds numbers

In [ ]:
Q_TUBE = (df["Q Tube In"] + df["Q Tube Out"]) / 2 / 1000     # L/s -> m^3/s
Q_SHELL = (df["Q Shell In"] + df["Q Shell Out"]) / 2 / 1000

df["u tube (m/s)"] = Q_TUBE / (N_FIB * np.pi * r_i**2)
df["u shell (m/s)"] = Q_SHELL / A_C
df["Re tube"] = D_I * df["u tube (m/s)"] / NU
df["Re shell"] = D_H * df["u shell (m/s)"] / NU

df[["run", "u tube (m/s)", "u shell (m/s)", "Re tube", "Re shell"]]

,run,u tube (m/s),u shell (m/s),Re tube,Re shell
0,1,0.007290,0.004543,1.632793,1.126064
1,2,0.007290,0.004429,1.632793,1.097912
2,3,0.007564,0.006587,1.694022,1.632793
3,4,0.007837,0.010790,1.755252,2.674402
4,5,0.007290,0.005792,1.632793,1.435731
5,6,0.007290,0.002044,1.632793,0.506729


Sherwood numbers

In [ ]:
df["Sh tube"] = 3.66 + df["Re tube"]**2.24 * (SC * D_I / L)**(1/3)
df["Sh shell"] = 1.39 * df["Re shell"]**2.11 * (SC * D_H / L)**(1/3)

df[["run", "Sh tube", "Sh shell"]]

,run,Sh tube,Sh shell
0,1,5.763700,1.295726
1,2,5.763700,1.228324
2,3,5.944532,2.837916
3,4,6.133653,8.038284
4,5,5.763700,2.163414
5,6,5.763700,0.240321


2c: BL coefficients

In [ ]:
df["k_RA (m/s)"] = df["Sh tube"] * D_AB / D_I
df["k_DA (m/s)"] = df["Sh shell"] * D_AB / D_H

df[["run", "k_RA (m/s)", "k_DA (m/s)"]]

,run,k_RA (m/s),k_DA (m/s)
0,1,0.000057,0.000012
1,2,0.000057,0.000011
2,3,0.000059,0.000026
3,4,0.000061,0.000072
4,5,0.000057,0.000019
5,6,0.000057,0.000002


2d: membrane coefficient

In [ ]:
resid = 1 / df["K_A (m/s)"] - 1 / df["k_RA (m/s)"] - r_i / (r_o * df["k_DA (m/s)"])
df["k_mA (m/s)"] = r_i * np.log(r_o / r_i) / ((r_o - r_i) * resid)

df[["run", "K_A (m/s)", "k_RA (m/s)", "k_DA (m/s)", "k_mA (m/s)"]]

,run,K_A (m/s),k_RA (m/s),k_DA (m/s),k_mA (m/s)
0,1,0.000005,0.000057,0.000012,0.000007
1,2,0.000005,0.000057,0.000011,0.000007
2,3,0.000006,0.000059,0.000026,0.000006
3,4,0.000006,0.000061,0.000072,0.000006
4,5,0.000006,0.000057,0.000019,0.000007
5,6,0.000003,0.000057,0.000002,-0.000036


3: epsilon/tau

In [ ]:
df["eps/tau"] = df["k_mA (m/s)"] * Z / D_AB


ok = df["eps/tau"] > 0
print(f"mean eps/tau (valid runs) = {df.loc[ok, 'eps/tau'].mean():.3f} "
      f"± {df.loc[ok, 'eps/tau'].std():.3f}")

df[["run","eps/tau"]]

mean eps/tau (valid runs) = 0.151 ± 0.011


,run,eps/tau
0,1,0.159257
1,2,0.160884
2,3,0.142436
3,4,0.136435
4,5,0.157015
5,6,-0.824243
